# Reading your data

Walks you through the FASTQ files sitting in your project's
analytical-dataset bucket. We'll list them, peek at one, count reads,
and confirm that paired-end files line up.

No pipeline launches in this notebook. Just inspection. Everything
else builds on what you'll see here.

Takes about three minutes. Most of that is streaming through the FASTQ
files (a hundred-odd MB each) to count records.

> **Kernel:** When JupyterLab prompts "Select Kernel," pick **`Python 3 (Local)`** (under "Start python Kernel"). Don't pick `Python 3 (ipykernel) (Local)`, PyTorch, or TensorFlow — those are missing libraries this notebook needs and will fail at the first GCS read with `ModuleNotFoundError`. If you already see `Python 3 (Local)` in the top-right of this tab, you're good to go.

In [ ]:
import subprocess

DETECTED_PROJECT = subprocess.check_output(
    ["gcloud", "config", "get-value", "project"], text=True
).strip()

# Defaults are auto-detected from the Workbench environment.
# Override these only if you want to point at something different.
PROJECT_ID  = DETECTED_PROJECT                # @param {type:"string"}
REGION      = "us-central1"                   # @param {type:"string"}
DATASET_URI = ""                              # @param {type:"string"}
# Leave DATASET_URI blank to auto-pick the first *-analytical-dataset-* bucket.

## Where am I?

Quick check on the environment.

In [ ]:
import subprocess

active_account = subprocess.check_output(
    ["gcloud", "config", "get-value", "account"], text=True
).strip()

print(f"GCP project: {PROJECT_ID}")
print(f"Region:      {REGION}")
print(f"Identity:    {active_account}")

## Setup: install Biopython

`gcsfs`, `pandas`, and the Google Cloud client libraries are already
in Vertex AI Workbench. We just need Biopython for parsing FASTQ
files. The cell below checks first and skips the install if it's
already there, so it's safe to re-run.

In [ ]:
import importlib
import site
import subprocess
import sys

try:
    importlib.import_module("Bio")
    print("Biopython is already installed.")
except ImportError:
    print("Installing Biopython via pip...")
    # --user writes to per-user site-packages, which works even if the
    # base env's site-packages is read-only (common on micromamba images).
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--user",
         "biopython"],
        check=True,
    )
    # Some kernels don't auto-include the user-site dir in sys.path if
    # it didn't exist at kernel startup (pip --user creates it on first
    # use). Add it explicitly so the import in the next cell finds Bio.
    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.insert(0, user_site)
    importlib.invalidate_caches()
    print("Biopython installed.")

## Your project's buckets

An APGAP project shows two kinds of bucket from inside the notebook.

`*-seqera-output-*` holds pipeline run outputs. Often empty if nothing's
been run yet.

`*-analytical-dataset-*` holds your input FASTQs, with one bucket per
dataset shown in the project's Datasets tab. The naming is a little
misleading: these are inputs to analysis, not curated analytical
results. A Nextflow pipeline would read these.

In [ ]:
import gcsfs

fs = gcsfs.GCSFileSystem()
if not DATASET_URI:
    all_analytical = [b.rstrip("/") for b in fs.ls("") if "analytical-dataset" in b]
    # Prefer buckets whose name starts with this project's pathogen prefix
    # (the leading token of PROJECT_ID, e.g. "h1n1" from "h1n1-apgap-project-0a47").
    pathogen = PROJECT_ID.split("-")[0]
    matched = [b for b in all_analytical if b.startswith(pathogen)]
    candidates = matched or all_analytical
    if not candidates:
        raise RuntimeError(
            "No analytical-dataset bucket visible. "
            "Open your project in the APGAP web app, go to the Datasets tab, "
            "copy a dataset URI, and paste it into DATASET_URI above."
        )
    DATASET_URI = f"gs://{candidates[0]}/"
    if len(all_analytical) > 1:
        others = [b for b in all_analytical if b != candidates[0]]
        print(f"Auto-selected dataset: {DATASET_URI}")
        print(f"  (other analytical-dataset buckets visible: {others})")
    else:
        print(f"Auto-selected dataset: {DATASET_URI}")
else:
    print(f"Using DATASET_URI: {DATASET_URI}")

## Files in this dataset

What's actually in the bucket. Every file, with size and content type.

In [ ]:
import gcsfs

fs = gcsfs.GCSFileSystem()
dataset_path = DATASET_URI.replace("gs://", "").rstrip("/")
files = sorted(fs.ls(dataset_path))

print(f"{'Size':>10}  {'Type':<20}  Path")
print(f"{'-'*10}  {'-'*20}  {'-'*40}")
for f in files:
    info = fs.info(f)
    size_mb = info.get("size", 0) / 1024 / 1024
    content_type = info.get("contentType", "—")
    print(f"{size_mb:>7.1f} MB  {content_type:<20}  gs://{f}")

## What's inside a FASTQ file?

FASTQ is gzipped text. Each read is 4 lines: an `@`-prefixed identifier,
the sequence, a `+` separator, then per-base quality scores. Paired-end
data splits a sample across `_1.fastq.gz` (forward) and `_2.fastq.gz`
(reverse), one record at a time matched by line position.

That's enough for the next cell to make sense. For the full format
reference — Phred quality encoding, ASCII offset rules, alternative
naming conventions — see [05-reference.ipynb](05-reference.ipynb).

Peek at one file the raw way. Stream the first few lines out of GCS and
print them:

In [ ]:
import gcsfs
import gzip
import io

fs = gcsfs.GCSFileSystem()
dataset_path = DATASET_URI.replace("gs://", "").rstrip("/")
fastq_files = sorted([f for f in fs.ls(dataset_path) if f.endswith(".fastq.gz")])

if not fastq_files:
    print(f"⚠ No .fastq.gz files found in {DATASET_URI}.")
    print("  The remaining cells in this notebook (and notebook 03) assume")
    print("  FASTQ files are present. Either point DATASET_URI at a dataset")
    print("  that contains some, or skip the rest of this notebook.")
else:
    first_file = f"gs://{fastq_files[0]}"
    print(f"Peeking at: {first_file}\n")

    with fs.open(fastq_files[0], "rb") as raw:
        with gzip.GzipFile(fileobj=raw) as gz:
            # Read first 8 lines = first 2 records
            for i, line in enumerate(gz):
                if i >= 8:
                    break
                print(line.decode().rstrip())

## A cleaner read with Biopython

Biopython knows FASTQ. Each record becomes a `SeqRecord` with `.id`,
`.seq`, and `.letter_annotations["phred_quality"]`. Easier than
parsing 4-line blocks by hand.

In [ ]:
import gcsfs
import gzip
import io

if not fastq_files:
    print("⚠ Skipping: no FASTQ files in this dataset (see warning above).")
else:
    from Bio import SeqIO
    import pandas as pd

    fs = gcsfs.GCSFileSystem()
    records = []
    with fs.open(fastq_files[0], "rb") as raw:
        with gzip.GzipFile(fileobj=raw) as gz:
            text = io.TextIOWrapper(gz, encoding="utf-8")
            for i, record in enumerate(SeqIO.parse(text, "fastq")):
                records.append({
                    "id": record.id,
                    "length": len(record.seq),
                    "min_qual": min(record.letter_annotations["phred_quality"]),
                    "max_qual": max(record.letter_annotations["phred_quality"]),
                    "preview": str(record.seq[:40]) + "...",
                })
                if i >= 2:
                    break

    pd.DataFrame(records)

## Basic stats per file

How many reads per file, and what's the average read length? Streaming
through 100-ish MB of gzipped data takes 20 to 30 seconds per file.
The cell prints a line as it starts each one so you can see it's
working.

In [ ]:
import gcsfs
import gzip
import io

if not fastq_files:
    print("⚠ Skipping: no FASTQ files to count (see warning above).")
    stats = []
else:
    from Bio import SeqIO

    fs = gcsfs.GCSFileSystem()
    stats = []
    for path in fastq_files:
        print(f"Counting {path}...", flush=True)
        n_reads = 0
        total_length = 0
        with fs.open(path, "rb") as raw:
            with gzip.GzipFile(fileobj=raw) as gz:
                text = io.TextIOWrapper(gz, encoding="utf-8")
                for record in SeqIO.parse(text, "fastq"):
                    n_reads += 1
                    total_length += len(record.seq)
        stats.append({
            "file": path.rsplit("/", 1)[-1],
            "n_reads": n_reads,
            "mean_length": round(total_length / n_reads, 1) if n_reads else 0,
            "size_mb": round(fs.info(path).get("size", 0) / 1024 / 1024, 1),
        })

    print("Done.")

In [ ]:
import pandas as pd
df = pd.DataFrame(stats)
df

## Paired-end sanity check

`*_1.fastq.gz` and `*_2.fastq.gz` files for the same sample should
have identical read counts. They're forward and reverse reads from
the same run, so each forward record has exactly one reverse partner.
A mismatch, or a missing direction, usually points to a corrupted or
truncated file.

If you see a mismatch: check the dataset in the APGAP web app to
confirm what was uploaded, then flag it to your project lead. Don't
run a pipeline on inconsistent paired-end data.

In [ ]:
import re

if not stats:
    print("⚠ Skipping: no FASTQ files to pair-check (see warning above).")
else:
    paired = {}
    for s in stats:
        fname = s["file"]
        m = re.match(r"(.+)_([12])\.fastq\.gz$", fname)
        if m:
            sample, direction = m.group(1), m.group(2)
            paired.setdefault(sample, {})[direction] = s["n_reads"]

    print(f"{'Sample':<25} {'R1 reads':>12} {'R2 reads':>12}  Match?")
    print(f"{'-'*25} {'-'*12} {'-'*12}  {'-'*6}")
    for sample, counts in sorted(paired.items()):
        r1, r2 = counts.get("1"), counts.get("2")
        if r1 is None or r2 is None:
            match = "✗ MISSING DIRECTION"
        elif r1 != r2:
            match = "✗ MISMATCH"
        else:
            match = "✓"
        print(f"{sample:<25} {r1!s:>12} {r2!s:>12}  {match}")

## Your dataset's metadata

APGAP keeps per-file metadata (sample ID, pathogen, collection
location, sequencing platform, and so on) as a CSV sidecar in the
dataset bucket. The file is named `_apgap_metadata.csv`; the leading
underscore is what hides it from the web app's Datasets view. It's
also what the web app reads to render the metadata page you see when
you click the eye icon on a sequence row.

The next cell loads it if it's there.

In [ ]:
import gcsfs
import pandas as pd
from IPython.display import display

fs = gcsfs.GCSFileSystem()
metadata_path = DATASET_URI.replace("gs://", "").rstrip("/") + "/_apgap_metadata.csv"

try:
    with fs.open(metadata_path, "r") as f:
        metadata = pd.read_csv(f)
    print(f"Loaded metadata for {len(metadata)} file(s) from {metadata_path}\n")
    display(metadata)
except FileNotFoundError:
    metadata = None
    print(f"No _apgap_metadata.csv found in {DATASET_URI}.")
    print("Older datasets may not have one; ask your project lead if you need the metadata.")

### Read counts + metadata in one view

Joining the metadata onto the per-file stats DataFrame from earlier
gives you one row per FASTQ with read counts AND sample metadata side
by side. Handy for confirming that what you think you have matches
what the lab recorded.

In [ ]:
from IPython.display import display

if metadata is None:
    print("Skipping enrichment because no metadata was loaded.")
elif df.empty:
    print("⚠ Metadata loaded but no per-file stats to join (no FASTQ files in dataset).")
else:
    # APGAP's metadata uses 'filename' as the join key.
    enriched = df.merge(metadata, left_on="file", right_on="filename", how="left")
    # Pick a few high-value columns to display first; everything else is in the DataFrame.
    preferred = [c for c in [
        "file", "n_reads", "mean_length", "size_mb",
        "SAMPLE ID", "PATHOGEN/ORGANISM NAME (OR METAGENOMIC)",
        "BIOSPECIMEN TYPE", "DATE COLLECTED", "SEQUENCING INSTRUMENT MAKE AND MODEL",
        "SEQUENCING LAB (ORIGINATING LAB)", "SOURCE LOCATION STATE",
    ] if c in enriched.columns]
    display(enriched[preferred])

## Where to go next

Now that you can see what's in your dataset:

- Launch a pipeline on it: [03-launch-a-pipeline.ipynb](03-launch-a-pipeline.ipynb)
- Pull more data from public archives: [04-download-from-public-repos.ipynb](04-download-from-public-repos.ipynb)
- File format reference (FASTQ, VCF, samplesheet, output layout): [05-reference.ipynb](05-reference.ipynb)
- Back to the [getting-started overview](01-getting-started.ipynb)

Background reading:

- [Biopython SeqIO docs](https://biopython.org/wiki/SeqIO)
- [FASTQ format on Wikipedia](https://en.wikipedia.org/wiki/FASTQ_format)